# Gate 1: generate candidates with a second model (GPU step)

This notebook only **generates** code. Labelling, pool building and all evaluation run on the laptop (CPU).

**Settings:** Accelerator **GPU T4 x1**, Internet **on**. For a private repo, add a Kaggle secret named `GITHUB_TOKEN` (*Add-ons → Secrets*).

**Then:** *Save Version → Save & Run All*. When it finishes, download `gen_<model>.zip` from the Output tab.

Prompt, seeds and decoding settings are the same as the Baseline 1 run (prompt hash `ebbe8bceae71`).

In [ ]:
# ---------------- Configuration ----------------
REPO_URL = "https://github.com/<user>/<repo>.git"   # <- your repo
BRANCH = "main"
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
N_SAMPLES = 5
TEMPERATURE = 0.7
LIMIT = None            # e.g. 3 for a quick dry run
TAG = MODEL_ID.split("/")[-1].lower().replace(".", "")
OUT = f"/kaggle/working/gen_{TAG}"
print(TAG, OUT)

In [ ]:
# ---------------- Get the code (no pip install: the kernel's torch/transformers are used as is) ----------------
import os, shutil, subprocess
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    print("no GITHUB_TOKEN secret; cloning without a token (public repo)")
os.chdir("/kaggle/working")
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)        # always a fresh clone of the latest push
subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "/kaggle/working/repo"], check=True)
os.chdir("/kaggle/working/repo")
print(subprocess.run(["git", "log", "-1", "--format=%H %s (%cd)"], capture_output=True, text=True).stdout)

NEEDED = ["src/igt/__init__.py", "src/igt/agents/__init__.py", "src/igt/agents/code_generator.py",
          "eval/generate.py", "eval/prompts/schema_b1-v1.json", "eval/tasks"]
missing = [f for f in NEEDED if not os.path.exists(f)]
assert not missing, f"These files are not in the GitHub repo (copy them from the workspace and push): {missing}"
print("all needed files present")

In [ ]:
# ---------------- Check the GPU and the prompt ----------------
import sys, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable the GPU accelerator")
sys.path.insert(0, "src")
from igt.agents.code_generator import PROMPT_HASH
assert PROMPT_HASH == "ebbe8bceae71", PROMPT_HASH
print("prompt hash OK:", PROMPT_HASH)

In [ ]:
# ---------------- Generate (resumable) ----------------
cmd = ["python", "eval/generate.py", "--model", MODEL_ID, "--out", OUT,
       "--n-samples", str(N_SAMPLES), "--temperature", str(TEMPERATURE)]
if LIMIT:
    cmd += ["--limit", str(LIMIT)]
p = subprocess.run(cmd, capture_output=True, text=True)
print(p.stdout[-3000:])
print(p.stderr[-3000:])
assert p.returncode == 0

In [ ]:
# ---------------- Quick look and zip ----------------
import json, shutil, collections
G = [json.loads(l) for l in open(f"{OUT}/generations.jsonl")]
print("generations:", len(G), "| truncated:", sum(g["truncated"] for g in G))
print("per split:", collections.Counter(g["split"] for g in G))
print(G[0]["raw_output"][:500])
open(f"{OUT}/git_commit.txt", "w").write(subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)
print("saved:", shutil.make_archive(f"/kaggle/working/gen_{TAG}", "zip", OUT))
os.chdir("/kaggle/working")
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)   # keep only the outputs